In [252]:
from unittest.mock import inplace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dbfread import DBF
import os

In [253]:
def read_dbf(filename):
	dbf = DBF(filename)
	df = pd.DataFrame(iter(dbf))

	df.replace(r"^\s*$", np.nan, inplace=True, regex=True)

	df.dropna(axis=1, how='all', inplace=True)
	df.dropna(axis=0, how='all', inplace=True)
	df.drop_duplicates(inplace=True)

	return df

In [254]:
def extend_countries_map(df, col):
	df_countries = pd.DataFrame(columns=['country_list_name'])
	if os.path.exists('./data/countries.csv'):
		df_countries = pd.read_csv('./data/countries.csv')
		df_countries.rename({'name': 'country_list_name'}, axis=1, inplace=True)

	# case-insensitive
	df['country_list_name'] = df[col].str.lower().str.strip()

	df_countries = pd.concat([df_countries, df[['country_list_name']]], axis=0)\
		.drop_duplicates(ignore_index=True)\
		.reset_index(names=f'{col}_index')

	df = df.merge(df_countries, how='left', on='country_list_name')
	df.drop([col, 'country_list_name'], axis=1, inplace=True)

	df_countries.drop(f'{col}_index', axis=1)\
		.rename({'country_list_name': 'name'}, axis=1)\
		.to_csv('./data/countries.csv', index=False)

	return df

In [255]:
df_exped = read_dbf('data/raw/exped.DBF')

In [256]:
df_exped.head(5)

,EXPID,PEAKID,YEAR,SEASON,HOST,ROUTE1,ROUTE2,ROUTE3,ROUTE4,NATION,...,ACCIDENTS,ACHIEVMENT,AGENCY,COMRTE,STDRTE,PRIMRTE,PRIMMEM,PRIMREF,PRIMID,CHKSUM
0,ANN260101,ANN2,1960,1,1,NW Ridge-W Ridge,NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2442047
1,ANN269301,ANN2,1969,3,1,NW Ridge-W Ridge,NaN,NaN,NaN,Yugoslavia,...,Draslar frostbitten hands and feet,NaN,NaN,None,None,False,False,None,NaN,2445501
2,ANN273101,ANN2,1973,1,1,W Ridge-N Face,NaN,NaN,NaN,Japan,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2446797
3,ANN278301,ANN2,1978,3,1,N Face-W Ridge,NaN,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2448822
4,ANN279301,ANN2,1979,3,1,N Face-W Ridge,NW Ridge of A-IV,NaN,NaN,UK,...,NaN,NaN,NaN,None,None,False,False,None,NaN,2449204


In [257]:
df_exped.shape

(11578, 66)

In [258]:
df_exped.BCDATE = pd.to_datetime(df_exped.BCDATE)
df_exped.SMTDATE = pd.to_datetime(df_exped.SMTDATE)
df_exped.TERMDATE = pd.to_datetime(df_exped.TERMDATE)

In [259]:
df_exped.HOST.value_counts()

HOST
1    9251
2    2256
3      69
0       2
Name: count, dtype: int64

In [260]:
host_map = {
	0: 'Unknown',
	1: 'Nepal',
	2: 'China',
	3: 'India'
}

In [261]:
df_exped.HOST = df_exped.HOST.map(host_map)
df_exped = extend_countries_map(df_exped,'HOST')

In [262]:
df_exped.NATION = df_exped.NATION.str.split("/")
df_exped = df_exped.explode('NATION')
df_exped = extend_countries_map(df_exped,'NATION')

In [263]:
df_exped.COUNTRIES = df_exped.COUNTRIES.str.split(",")
df_exped = df_exped.explode('COUNTRIES')

df_exped.COUNTRIES = df_exped.COUNTRIES.str.split("/")
df_exped = df_exped.explode('COUNTRIES')

df_exped = extend_countries_map(df_exped,'COUNTRIES')

In [264]:
df_exped.drop([
	'LEADERS', 'SMTTIME', 'TERMNOTE', 'OTHERSMTS', 'CAMPSITES', 'ROUTEMEMO', 'HIGHPOINT', 'ACCIDENTS', 'ACHIEVMENT',
	'PRIMMEM', 'PRIMREF', 'PRIMID', 'CHKSUM'], axis=1, inplace=True)

In [265]:
df_exped.head()

,EXPID,PEAKID,YEAR,SEASON,ROUTE1,ROUTE2,ROUTE3,ROUTE4,SPONSOR,SUCCESS1,...,O2MEDICAL,O2TAKEN,O2UNKWN,AGENCY,COMRTE,STDRTE,PRIMRTE,HOST_index,NATION_index,COUNTRIES_index
0,ANN260101,ANN2,1960,1,NW Ridge-W Ridge,NaN,NaN,NaN,NaN,True,...,False,False,False,NaN,None,None,False,0,4,2
1,ANN260101,ANN2,1960,1,NW Ridge-W Ridge,NaN,NaN,NaN,NaN,True,...,False,False,False,NaN,None,None,False,0,4,0
2,ANN269301,ANN2,1969,3,NW Ridge-W Ridge,NaN,NaN,NaN,Mountaineering Club of Slovenia,True,...,False,False,False,NaN,None,None,False,0,5,98
3,ANN273101,ANN2,1973,1,W Ridge-N Face,NaN,NaN,NaN,Sangaku Doshikai Annapurna II Expedition 1973,True,...,False,False,False,NaN,None,None,False,0,6,98
4,ANN278301,ANN2,1978,3,N Face-W Ridge,NaN,NaN,NaN,British Annapurna II Expedition,False,...,False,False,False,NaN,None,None,False,0,4,98
